# SFT Paper 4 Reproducibility Notebook

This notebook is aligned with the latest uploaded Paper 4 draft.

It reproduces:

1. \(S^3\) scalar/vector eigenvalue ladder.
2. Geometric scales \(M_{\rm surf}\) and \(E_{\rm vac}\).
3. Operator \(\rightarrow\) eigenvalue \(\rightarrow\) classification \(\rightarrow\) mass chain.
4. Charged-lepton masses.
5. Quark masses.
6. Preliminary neutrino closure sequence.
7. Higgs status.
8. W/Z status.

W and Z are not computed because the latest uploaded draft does not provide explicit SFT mass formulas for them.


## 1. Load constants and compute geometric scales

In [ ]:
from fractions import Fraction
import math
import pandas as pd

# SI constants
C_LIGHT = 299_792_458.0
HBAR = 1.054_571_817e-34
G_NEWTON = 6.674_30e-11
EV_J = 1.602_176_634e-19
J_PER_MEV = EV_J * 1.0e6

# SFT baseline values used by the latest draft
R_C_M = 1.4466e27
RHO_DE0_KG_M3 = 5.95765e-27

def planck_length_m():
    return math.sqrt(HBAR * G_NEWTON / C_LIGHT**3)

def surface_mass_scale_mev(R_c_m=R_C_M):
    l_p = planck_length_m()
    energy_j = (HBAR * C_LIGHT / R_c_m) * (R_c_m / l_p)**(2/3)
    return energy_j / J_PER_MEV

def vacuum_energy_scale_mev(rho_de0=RHO_DE0_KG_M3):
    energy_j = (rho_de0 * C_LIGHT**2 * (HBAR * C_LIGHT)**3)**0.25
    return energy_j / J_PER_MEV

M_SURF_MEV = surface_mass_scale_mev()
E_VAC_MEV = vacuum_energy_scale_mev()

scales_df = pd.DataFrame([
    {"Quantity": "Planck length", "Value": planck_length_m(), "Unit": "m"},
    {"Quantity": "M_surf", "Value": M_SURF_MEV, "Unit": "MeV"},
    {"Quantity": "E_vac", "Value": E_VAC_MEV, "Unit": "MeV"},
    {"Quantity": "E_vac", "Value": E_VAC_MEV * 1e9, "Unit": "meV"},
])
scales_df


## 2. Reproduce \(S^3\) eigenvalue ladder

Scalar:
\[
\widetilde{\Lambda}^{(0)}_n=n(n+2).
\]

Vector:
\[
\widetilde{\Lambda}^{(1)}_n=n(n+2)-1.
\]


In [ ]:
def scalar_lambda_tilde(n):
    return n * (n + 2)

def vector_lambda_tilde(n):
    return n * (n + 2) - 1

def scalar_degeneracy(n):
    return (n + 1)**2

def vector_degeneracy(n):
    return 2 * n * (n + 2)

manuscript_ladder = pd.DataFrame([
    {"Sector": "Scalar", "n": 1, "Lambda_tilde": scalar_lambda_tilde(1), "Degeneracy": scalar_degeneracy(1), "Interpretation": "Low-order scalar sector"},
    {"Sector": "Vector", "n": 1, "Lambda_tilde": vector_lambda_tilde(1), "Degeneracy": vector_degeneracy(1), "Interpretation": "Low-order vector sector"},
    {"Sector": "Vector", "n": 2, "Lambda_tilde": vector_lambda_tilde(2), "Degeneracy": vector_degeneracy(2), "Interpretation": "Higher vector sector"},
    {"Sector": "Vector", "n": 3, "Lambda_tilde": vector_lambda_tilde(3), "Degeneracy": vector_degeneracy(3), "Interpretation": "Higher localized vector sector"},
    {"Sector": "Scalar", "n": 4, "Lambda_tilde": scalar_lambda_tilde(4), "Degeneracy": scalar_degeneracy(4), "Interpretation": "Higher localized scalar sector"},
])
manuscript_ladder


## 3. Operator to eigenvalue to classification

In [ ]:
def classification_element(a, b, c):
    value = Fraction(1, 1)
    for base, power in [(2, a), (3, b), (4, c)]:
        value *= Fraction(base**power, 1) if power >= 0 else Fraction(1, base**(-power))
    return value

retained = [
    ("e", "charged lepton", "Vector", 1, "Lowest retained", 1, 1, (-1, -3, 0)),
    ("mu", "charged lepton", "Vector", 2, "First excitation", 1, 1, (0, 0, 1)),
    ("tau", "charged lepton", "Vector", 3, "Higher excitation", 1, 1, (0, 0, 3)),
    ("u", "quark", "Vector", 1, "Lowest retained", 3, 1, (0, -1, -1)),
    ("d", "quark", "Vector", 1, "Lowest retained", 3, 1, (1, -1, -1)),
    ("s", "quark", "Vector", 2, "First excitation", 3, 1, (0, 0, 1)),
    ("c", "quark", "Vector", 3, "Higher excitation", 3, 1, (0, 1, 2)),
    ("b", "quark", "Vector", 3, "Higher excitation", 3, 1, (0, 2, 2)),
    ("t", "quark", "Vector", 4, "Maximal retained", 8, 1, (0, 8, 0)),
]

rows = []
for particle, sector, op_sector, n, role, d_retained, r_i, abc in retained:
    lam = vector_lambda_tilde(n)
    raw_deg = vector_degeneracy(n)
    C = classification_element(*abc)
    m_mev = float(C) * M_SURF_MEV
    rows.append({
        "Particle": particle,
        "Sector": sector,
        "Operator sector": op_sector,
        "Mode n": n,
        "Lambda_tilde": lam,
        "Raw harmonic degeneracy": raw_deg,
        "Lambda role": role,
        "d_i retained": d_retained,
        "r_i": r_i,
        "(a,b,c)": abc,
        "C_i exact": str(C),
        "C_i decimal": float(C),
        "M_surf_MeV": M_SURF_MEV,
        "m_SFT_MeV": m_mev,
        "m_SFT_GeV": m_mev / 1000,
    })

retained_df = pd.DataFrame(rows)
retained_df


## 4. Compare with observations

In [ ]:
observed = {
    "e": 0.51099895000,
    "mu": 105.6583755,
    "tau": 1776.86,
    "u": 2.16,
    "d": 4.67,
    "s": 93.4,
    "c": 1270.0,
    "b": 4180.0,
    "t": 172570.0,
}

comparison = []
for _, row in retained_df.iterrows():
    p = row["Particle"]
    pred = row["m_SFT_MeV"]
    obs = observed[p]
    comparison.append({
        "Particle": p,
        "m_SFT_MeV": round(pred, 6),
        "Observed_MeV": obs,
        "Absolute_Error_MeV": round(pred - obs, 6),
        "Percent_Error": round(100*(pred - obs)/obs, 4),
        "Comment": "running mass; scale-dependent" if p in {"u","d","s","c","b","t"} else "",
    })
comparison_df = pd.DataFrame(comparison)
comparison_df


## 5. Preliminary neutrino closure sequence

In [ ]:
neutrino_df = pd.DataFrame([
    {"Particle": "nu1", "Factor": 0, "E_vac_MeV": E_VAC_MEV, "m_SFT_MeV": 0.0},
    {"Particle": "nu2", "Factor": 4, "E_vac_MeV": E_VAC_MEV, "m_SFT_MeV": 4 * E_VAC_MEV},
    {"Particle": "nu3", "Factor": 24, "E_vac_MeV": E_VAC_MEV, "m_SFT_MeV": 24 * E_VAC_MEV},
])
neutrino_df["m_SFT_eV"] = neutrino_df["m_SFT_MeV"] * 1e6
neutrino_df["m_SFT_meV"] = neutrino_df["m_SFT_MeV"] * 1e9
neutrino_df["Status"] = "preliminary neutrino closure sequence"
neutrino_df


## 6. W, Z, and Higgs status

In [ ]:
HIGGS_SFT_GEV_REPORTED = 123.8
OBSERVED_H_GEV = 125.20

ew_status = pd.DataFrame([
    {"Particle": "W", "m_SFT_GeV": None, "Observed_GeV": 80.377, "Status": "not computed: explicit SFT W formula not provided in latest draft"},
    {"Particle": "Z", "m_SFT_GeV": None, "Observed_GeV": 91.1876, "Status": "not computed: explicit SFT Z formula not provided in latest draft"},
    {"Particle": "H", "m_SFT_GeV": HIGGS_SFT_GEV_REPORTED, "Observed_GeV": OBSERVED_H_GEV, "Status": "reported candidate; scalar closure correction not fully derived"},
])
ew_status["Absolute_Error_GeV"] = ew_status.apply(lambda r: None if pd.isna(r["m_SFT_GeV"]) else r["m_SFT_GeV"] - r["Observed_GeV"], axis=1)
ew_status["Percent_Error"] = ew_status.apply(lambda r: None if pd.isna(r["m_SFT_GeV"]) else 100*(r["m_SFT_GeV"] - r["Observed_GeV"])/r["Observed_GeV"], axis=1)
ew_status


## 7. Export CSV files

In [ ]:
from pathlib import Path
out = Path("outputs")
out.mkdir(exist_ok=True)

scales_df.to_csv(out / "SFT_Paper4_scales.csv", index=False)
manuscript_ladder.to_csv(out / "SFT_Paper4_manuscript_eigenvalue_table.csv", index=False)
retained_df.to_csv(out / "SFT_Paper4_operator_to_classification_to_mass.csv", index=False)
comparison_df.to_csv(out / "SFT_Paper4_observed_comparison.csv", index=False)
neutrino_df.to_csv(out / "SFT_Paper4_neutrino_closure_sequence.csv", index=False)
ew_status.to_csv(out / "SFT_Paper4_electroweak_higgs_status.csv", index=False)

print("CSV files saved in ./outputs")
